<a href="https://colab.research.google.com/github/Minakshi654/Modelname/blob/main/Text%20classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

class AnomalyClassifier:
    def __init__(self, category_examples: dict):
        """
        category_examples: dict mapping category -> list of example anomalies
        """
        self.data = []
        for category, examples in category_examples.items():
            for ex in examples:
                self.data.append((ex, category))

        # Create DataFrame
        self.df = pd.DataFrame(self.data, columns=["anomaly", "category"])

        # Build TF-IDF model
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self.tfidf_matrix = self.vectorizer.fit_transform(self.df["anomaly"])

    def classify(self, user_input: str) -> str:
        """
        Classify a custom anomaly description into the most relevant category.
        """
        user_vec = self.vectorizer.transform([user_input])
        similarities = cosine_similarity(user_vec, self.tfidf_matrix).flatten()
        best_match_idx = similarities.argmax()

        if similarities[best_match_idx] < 0.2:  # too different
            return "Category not found (too different from known examples)"

        matched_category = self.df.iloc[best_match_idx]["category"]
        return f"Predicted Category: {matched_category}"


# ---------------- Example Usage ----------------

# 1. Define categories with a few sample anomalies each
category_examples = {
    "Transaction & Value Integrity": [
        "Receivables amount = 0 or negative",
        "Outliers in receivable values",
        "Duplicate billing amounts"
    ],
    "Date & Timeline Consistency": [
        "Posting date in the future",
        "Net due date before posting date",
        "Month-end spike in transactions"
    ],
    "Customer & Master Data Accuracy": [
        "Same customer ID but different names",
        "Inactive customers billed",
        "Customer cluster mismatch"
    ],
    "Reference Codes & Structural Validity": [
        "Terms of Payment Key not aligned with Days",
        "Duplicate billing document numbers",
        "FI Document Type mismatch"
    ],
    "Behavioral & Pattern Deviations": [
        "Splitting invoices to avoid approval",
        "Unusual receivable spikes",
        "Overuse of generic reason codes"
    ]
}

# 2. Create classifier
classifier = AnomalyClassifier(category_examples)

# 3. Classify custom anomalies
print(classifier.classify("Payment key mismatch with terms"))
print(classifier.classify("Future posting beyond today"))
print(classifier.classify("Invoices broken into smaller chunks to avoid checks"))
print(classifier.classify("Customer appears with different names in system"))


Predicted Category: Reference Codes & Structural Validity
Predicted Category: Date & Timeline Consistency
Predicted Category: Behavioral & Pattern Deviations
Predicted Category: Customer & Master Data Accuracy


In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os

class ExcelAnomalyClassifier:
    def __init__(self, excel_file="anomaly_mapping.xlsx"):
        self.excel_file = excel_file
        self._load_data()

    def _load_data(self):
        if os.path.exists(self.excel_file):
            self.df = pd.read_excel(self.excel_file)
        else:
            # create empty file if doesn't exist
            self.df = pd.DataFrame(columns=["anomaly", "category"])
            self.df.to_excel(self.excel_file, index=False)
        self._fit_model()

    def _fit_model(self):
        if len(self.df) > 0:
            self.vectorizer = TfidfVectorizer(stop_words="english")
            self.tfidf_matrix = self.vectorizer.fit_transform(self.df["anomaly"])
        else:
            self.vectorizer, self.tfidf_matrix = None, None

    def classify(self, user_input: str, top_k: int = 3):
        if self.tfidf_matrix is None:
            return []

        user_vec = self.vectorizer.transform([user_input])
        similarities = cosine_similarity(user_vec, self.tfidf_matrix).flatten()
        best_indices = similarities.argsort()[-top_k:][::-1]

        suggestions = []
        for idx in best_indices:
            if similarities[idx] > 0.1:
                suggestions.append((self.df.iloc[idx]["anomaly"],
                                     self.df.iloc[idx]["category"],
                                     round(similarities[idx], 3)))
        return suggestions

    def update(self, anomaly: str, category: str):
        # check for duplicates
        if ((self.df["anomaly"] == anomaly) & (self.df["category"] == category)).any():
            print("Already exists, not appending.")
            return

        # append new anomaly
        new_row = pd.DataFrame([[anomaly, category]], columns=["anomaly", "category"])
        self.df = pd.concat([self.df, new_row]).drop_duplicates()
        self.df.to_excel(self.excel_file, index=False)
        self._fit_model()
        print(f"Appended: {anomaly} → {category}")


# ---------------- Example Usage ----------------

classifier = ExcelAnomalyClassifier("anomaly_mapping.xlsx")

user_input = "Future posting beyond today"  # Example custom anomaly
print("User Input:", user_input)

# Step 1: Suggest categories
suggestions = classifier.classify(user_input, top_k=3)
print("\nSuggested Categories:")
for s in suggestions:
    print(f"Match: {s[0]} | Category: {s[1]} | Score: {s[2]}")

# Step 2: Simulate user validation (replace with actual input in practice)
correct_category = "Date & Timeline Consistency"
classifier.update(user_input, correct_category)


User Input: Future posting beyond today

Suggested Categories:
Appended: Future posting beyond today → Date & Timeline Consistency


In [4]:
import pandas as pd

# Define the anomalies and their categories
anomalies = [
    # Transaction & Value Integrity
    ("Receivables amount = 0 or negative", "Transaction & Value Integrity"),
    ("Receivables unusually high vs. historical average", "Transaction & Value Integrity"),
    ("Same customer billed multiple times with duplicate amounts", "Transaction & Value Integrity"),
    ("Currency mismatch", "Transaction & Value Integrity"),
    ("Outliers in receivable values", "Transaction & Value Integrity"),
    ("Posting key inconsistent with receivable amount", "Transaction & Value Integrity"),
    ("Receivable amount not aligned with payment terms", "Transaction & Value Integrity"),
    ("Backdated receivables before contract creation", "Transaction & Value Integrity"),
    ("Receivables posted after net due date", "Transaction & Value Integrity"),
    ("High-value transactions but missing reason code", "Transaction & Value Integrity"),

    # Date & Timeline Consistency
    ("Posting date in the future", "Date & Timeline Consistency"),
    ("Net due date before posting date", "Date & Timeline Consistency"),
    ("Net due date inconsistent with Terms of Payment Key (Days)", "Date & Timeline Consistency"),
    ("Different fiscal years for posting date vs company code", "Date & Timeline Consistency"),
    ("Weekends/holidays used as posting date", "Date & Timeline Consistency"),
    ("Unusually clustered transactions at midnight timestamps", "Date & Timeline Consistency"),
    ("Backdated posting after receivable already settled", "Date & Timeline Consistency"),
    ("Month-end spike in transactions", "Date & Timeline Consistency"),
    ("Net due date same as posting date (except cash terms)", "Date & Timeline Consistency"),
    ("Leap-year bug (29th Feb issues)", "Date & Timeline Consistency"),

    # Customer & Master Data Accuracy
    ("Same customer ID but different names/texts", "Customer & Master Data Accuracy"),
    ("Duplicate customers across different groups", "Customer & Master Data Accuracy"),
    ("Customers missing segment/region assignment", "Customer & Master Data Accuracy"),
    ("Customers assigned to multiple collection specialists", "Customer & Master Data Accuracy"),
    ("Customers belonging to two profit centers simultaneously", "Customer & Master Data Accuracy"),
    ("Customer cluster not matching customer segment", "Customer & Master Data Accuracy"),
    ("New customer with unusually large receivables", "Customer & Master Data Accuracy"),
    ("Customers in wrong country trading partner", "Customer & Master Data Accuracy"),
    ("Inactive customers billed", "Customer & Master Data Accuracy"),
    ("RB Customer (external) mismatched with internal ID", "Customer & Master Data Accuracy"),

    # Reference Codes & Structural Validity
    ("Terms of Payment Key not aligned with Terms of Payment Key (Days)", "Reference Codes & Structural Validity"),
    ("Customers assigned terms contradicting region’s default", "Reference Codes & Structural Validity"),
    ("Collection Group inconsistent with Collection Specialist", "Reference Codes & Structural Validity"),
    ("Collection Specialist handling too many high-value accounts", "Reference Codes & Structural Validity"),
    ("Payment terms changed after posting", "Reference Codes & Structural Validity"),
    ("Same Terms of Payment Key Text for multiple key codes", "Reference Codes & Structural Validity"),
    ("Unrealistic payment terms (e.g., 999 days)", "Reference Codes & Structural Validity"),
    ("Prepayment customers billed under post-payment terms", "Reference Codes & Structural Validity"),
    ("High overdue invoices with missing reason code", "Reference Codes & Structural Validity"),
    ("Collection group = “agency” but no follow-up invoices", "Reference Codes & Structural Validity"),
    ("FI Document Type does not match Sales Document Type", "Reference Codes & Structural Validity"),
    ("Reason code group text and code mismatched", "Reference Codes & Structural Validity"),
    ("Same reason code used for different reasons", "Reference Codes & Structural Validity"),
    ("Duplicate billing document numbers", "Reference Codes & Structural Validity"),
    ("Trading Partner AR text not matching ID", "Reference Codes & Structural Validity"),
    ("KPI Customer Cluster inconsistent with KPI Customer", "Reference Codes & Structural Validity"),
    ("Sales organization not mapped to correct region", "Reference Codes & Structural Validity"),
    ("Company code mismatch with profit center", "Reference Codes & Structural Validity"),
    ("Posting key not valid for FI Document Type", "Reference Codes & Structural Validity"),
    ("Missing link between order reason and receivable entry", "Reference Codes & Structural Validity"),

    # Behavioral & Pattern Deviations
    ("Customers with sudden increase in receivables", "Behavioral & Pattern Deviations"),
    ("Collection Specialist unusually active in multiple regions", "Behavioral & Pattern Deviations"),
    ("Invoices split into smaller amounts to avoid approval thresholds", "Behavioral & Pattern Deviations"),
    ("Same receivable amounts repeating across many customers", "Behavioral & Pattern Deviations"),
    ("Receivables always adjusted by same specialist", "Behavioral & Pattern Deviations"),
    ("Rounding errors (e.g., 0.01 differences consistently)", "Behavioral & Pattern Deviations"),
    ("Missing seasonal billing patterns (monthly contracts skipped)", "Behavioral & Pattern Deviations"),
    ("Customer receivables paid but still marked open", "Behavioral & Pattern Deviations"),
    ("Duplicate net due dates across unrelated customers", "Behavioral & Pattern Deviations"),
    ("Overuse of generic reason codes (“other”, “manual”, etc.)", "Behavioral & Pattern Deviations")
]

# Create DataFrame and save to CSV
df_mapping = pd.DataFrame(anomalies, columns=['Anomaly', 'Category'])
df_mapping.to_csv('anomaly_category_mapping.csv', index=False)
df_mapping.head()  # Display first few rows to verify creation


,Anomaly,Category
0,Receivables amount = 0 or negative,Transaction & Value Integrity
1,Receivables unusually high vs. historical average,Transaction & Value Integrity
2,Same customer billed multiple times with dupli...,Transaction & Value Integrity
3,Currency mismatch,Transaction & Value Integrity
4,Outliers in receivable values,Transaction & Value Integrity


In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Step 1: Load the billing data
data_file = 'dummy_billing_data.csv'
df = pd.read_csv(data_file)

# Step 2: Load anomaly-category mapping from CSV
mapping_file = 'anomaly_category_mapping.csv'
try:
    mapping_df = pd.read_csv(mapping_file)
except FileNotFoundError:
    mapping_df = pd.DataFrame(columns=['Anomaly', 'Category'])

# Step 3: Ask user for anomaly input
custom_anomaly = input('Enter your anomaly description: ').strip()

# Step 4: Check if exact match exists
exact_match = mapping_df[mapping_df['Anomaly'].str.lower() == custom_anomaly.lower()]
if not exact_match.empty:
    exact_category = exact_match.iloc[0]['Category']
    print(f'Exact anomaly found. Category: {exact_category}')
else:
    # Flatten for ML matching
    anomaly_texts = mapping_df['Anomaly'].tolist()
    categories_list = mapping_df['Category'].tolist()

    # TF-IDF Vectorizer
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(anomaly_texts)

    # Predict category using cosine similarity
    custom_vec = vectorizer.transform([custom_anomaly])
    similarity = cosine_similarity(custom_vec, X)
    top_indices = similarity[0].argsort()[-2:][::-1]  # Top 2 recommendations
    print('\nRecommended categories for your anomaly:')
    for idx in top_indices:
        print(f'- {categories_list[idx]} (similar example: {anomaly_texts[idx]})')

    # Step 5: Ask user to confirm the correct category
    correct_category = input('\nEnter the correct category from the recommended ones: ').strip()

    # Step 6: Append custom anomaly to CSV if not duplicate
    if not ((mapping_df['Anomaly'].str.lower() == custom_anomaly.lower()) & (mapping_df['Category'].str.lower() == correct_category.lower())).any():
        new_row = pd.DataFrame({'Anomaly': [custom_anomaly], 'Category': [correct_category]})
        mapping_df = pd.concat([mapping_df, new_row], ignore_index=True)
        mapping_df.to_csv(mapping_file, index=False)
        print(f'Custom anomaly added to category: {correct_category}')
    else:
        print('Custom anomaly already exists in the category.')


Enter your anomaly description: Payment terms changed after posting
Exact anomaly found. Category: Reference Codes & Structural Validity


In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Step 1: Load the billing data
data_file = 'dummy_billing_data.csv'
df = pd.read_csv(data_file)

# Step 2: Load anomaly-category mapping from CSV
mapping_file = 'anomaly_category_mapping.csv'
try:
    mapping_df = pd.read_csv(mapping_file)
except FileNotFoundError:
    mapping_df = pd.DataFrame(columns=['Anomaly', 'Category'])

# Step 3: Loop to ask for multiple anomalies
while True:
    custom_anomaly = input('\nEnter your anomaly description: ').strip()

    # Step 4: Check if exact match exists
    exact_match = mapping_df[mapping_df['Anomaly'].str.lower() == custom_anomaly.lower()]
    if not exact_match.empty:
        exact_category = exact_match.iloc[0]['Category']
        print(f'Exact anomaly found. Category: {exact_category}')
    else:
        # Flatten for ML matching
        anomaly_texts = mapping_df['Anomaly'].tolist()
        categories_list = mapping_df['Category'].tolist()

        # TF-IDF Vectorizer
        vectorizer = TfidfVectorizer()
        X = vectorizer.fit_transform(anomaly_texts)

        # Predict category using cosine similarity
        custom_vec = vectorizer.transform([custom_anomaly])
        similarity = cosine_similarity(custom_vec, X)
        top_indices = similarity[0].argsort()[-2:][::-1]  # Top 2 recommendations
        print('\nRecommended categories for your anomaly:')
        for idx in top_indices:
            print(f'- {categories_list[idx]} (similar example: {anomaly_texts[idx]})')

        # Step 5: Ask user to confirm the correct category
        correct_category = input('\nEnter the correct category from the recommended ones: ').strip()

        # Step 6: Append custom anomaly to CSV if not duplicate
        if not ((mapping_df['Anomaly'].str.lower() == custom_anomaly.lower()) & (mapping_df['Category'].str.lower() == correct_category.lower())).any():
            new_row = pd.DataFrame({'Anomaly': [custom_anomaly], 'Category': [correct_category]})
            mapping_df = pd.concat([mapping_df, new_row], ignore_index=True)
            mapping_df.to_csv(mapping_file, index=False)
            print(f'Custom anomaly added to category: {correct_category}')
        else:
            print('Custom anomaly already exists in the category.')

    # Step 7: Ask if done or continue
    done = input('\nDo you want to enter another anomaly? (yes/no): ').strip().lower()
    if done != 'yes':
        print('Exiting anomaly input.')
        break



Enter your anomaly description: Collection Specialist unusually active in multiple regions
Exact anomaly found. Category: Behavioral & Pattern Deviations

Do you want to enter another anomaly? (yes/no): yes

Enter your anomaly description: Receivables linked to cancelled orders

Recommended categories for your anomaly:
- Customer & Master Data Accuracy (similar example: Customers assigned to multiple collection specialists)
- Reference Codes & Structural Validity (similar example: Sales organization not mapped to correct region)

Enter the correct category from the recommended ones: Transaction & Value Integrity
Custom anomaly added to category: Transaction & Value Integrity

Do you want to enter another anomaly? (yes/no): yes

Enter your anomaly description: Inconsistent collection specialist assignment

Recommended categories for your anomaly:
- Reference Codes & Structural Validity (similar example: Collection Group inconsistent with Collection Specialist)
- Customer & Master Data 

In [9]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load anomaly-category mapping
mapping_file = 'anomaly_category_mapping.csv'
try:
    mapping_df = pd.read_csv(mapping_file)
except FileNotFoundError:
    mapping_df = pd.DataFrame(columns=['Anomaly', 'Category'])

# Initialize TF-IDF vectorizer for anomaly texts
vectorizer = TfidfVectorizer()

while True:
    custom_anomaly = input('\nEnter your anomaly description: ').strip()

    # Check if exact anomaly exists
    exact_match = mapping_df[mapping_df['Anomaly'].str.lower() == custom_anomaly.lower()]
    if not exact_match.empty:
        exact_category = exact_match.iloc[0]['Category']
        print(f'Exact anomaly found. Category: {exact_category}')
    else:
        # Step 1: Recommend category
        anomaly_texts = mapping_df['Anomaly'].tolist()
        categories_list = mapping_df['Category'].tolist()

        if len(anomaly_texts) == 0:
            print("No existing anomalies to compare. Please add one manually.")
            correct_category = input('Enter the correct category for your anomaly: ').strip()
        else:
            # Vectorize and compute similarity
            X = vectorizer.fit_transform(anomaly_texts)
            custom_vec = vectorizer.transform([custom_anomaly])
            similarity = cosine_similarity(custom_vec, X)

            # Find the most similar anomaly and its category
            top_idx = similarity[0].argsort()[-1]  # Most similar
            suggested_category = categories_list[top_idx]
            print(f'\nSuggested category for your anomaly: {suggested_category}')

            # Step 2: Recommend similar anomalies within suggested category
            filtered_df = mapping_df[mapping_df['Category'] == suggested_category]
            filtered_texts = filtered_df['Anomaly'].tolist()
            if len(filtered_texts) > 0:
                filtered_vecs = vectorizer.fit_transform(filtered_texts)
                sim_filtered = cosine_similarity(custom_vec, filtered_vecs)
                top_sim_indices = sim_filtered[0].argsort()[-5:][::-1]

                print('\nTop 5 similar anomalies in this category:')
                for i, idx in enumerate(top_sim_indices, start=1):
                    print(f'{i}. "{filtered_texts[idx]}"')
            correct_category = input('\nConfirm or modify the category for your anomaly: ').strip()

        # Step 3: Append anomaly if not duplicate
        if not ((mapping_df['Anomaly'].str.lower() == custom_anomaly.lower()) &
                (mapping_df['Category'].str.lower() == correct_category.lower())).any():
            new_row = pd.DataFrame({'Anomaly': [custom_anomaly], 'Category': [correct_category]})
            mapping_df = pd.concat([mapping_df, new_row], ignore_index=True)
            mapping_df.to_csv(mapping_file, index=False)
            print(f'Custom anomaly added to category: {correct_category}')
        else:
            print('Custom anomaly already exists in the category.')

        # Step 4: Collect feedback
        feedback = input('Were these recommendations helpful? (yes/no): ').strip().lower()
        if feedback == 'yes':
            print('Thanks for your feedback!')
        else:
            print('Thanks for your feedback. We will improve future suggestions.')

    # Continue loop?
    done = input('\nDo you want to enter another anomaly? (yes/no): ').strip().lower()
    if done != 'yes':
        print('Exiting anomaly input.')
        break



Enter your anomaly description: Invoices posted with inconsistent currency exchange rates

Suggested category for your anomaly: Transaction & Value Integrity


ValueError: Incompatible dimension for X and Y matrices: X.shape[1] == 193 while Y.shape[1] == 49

In [8]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load the billing data (if needed for other purposes)
data_file = 'dummy_billing_data.csv'
df = pd.read_csv(data_file)

# Load anomaly-category mapping
mapping_file = 'anomaly_category_mapping.csv'
try:
    mapping_df = pd.read_csv(mapping_file)
except FileNotFoundError:
    mapping_df = pd.DataFrame(columns=['Anomaly', 'Category'])

# Initialize TF-IDF vectorizer outside the loop for efficiency
vectorizer = TfidfVectorizer()

while True:
    custom_anomaly = input('\nEnter your anomaly description: ').strip()

    # Exact match check
    exact_match = mapping_df[mapping_df['Anomaly'].str.lower() == custom_anomaly.lower()]
    if not exact_match.empty:
        exact_category = exact_match.iloc[0]['Category']
        print(f'Exact anomaly found. Category: {exact_category}')
    else:
        # Vectorize anomalies and compute similarities
        anomaly_texts = mapping_df['Anomaly'].tolist()
        categories_list = mapping_df['Category'].tolist()

        if len(anomaly_texts) == 0:
            print("No existing anomalies to compare. Please add one manually.")
            correct_category = input('Enter the correct category for your anomaly: ').strip()
            new_row = pd.DataFrame({'Anomaly': [custom_anomaly], 'Category': [correct_category]})
            mapping_df = pd.concat([mapping_df, new_row], ignore_index=True)
            mapping_df.to_csv(mapping_file, index=False)
            print(f'Custom anomaly added to category: {correct_category}')
        else:
            X = vectorizer.fit_transform(anomaly_texts)
            custom_vec = vectorizer.transform([custom_anomaly])
            similarity = cosine_similarity(custom_vec, X)

            # Top 5 recommendations
            top_indices = similarity[0].argsort()[-5:][::-1]

            print('\nTop 5 recommended similar anomalies:')
            for i, idx in enumerate(top_indices, start=1):
                print(f'{i}. "{anomaly_texts[idx]}" → Category: {categories_list[idx]}')

            # Ask user which category is correct (or add new)
            correct_category = input('\nEnter the correct category for your anomaly: ').strip()

            # Append if not duplicate
            if not ((mapping_df['Anomaly'].str.lower() == custom_anomaly.lower()) &
                    (mapping_df['Category'].str.lower() == correct_category.lower())).any():
                new_row = pd.DataFrame({'Anomaly': [custom_anomaly], 'Category': [correct_category]})
                mapping_df = pd.concat([mapping_df, new_row], ignore_index=True)
                mapping_df.to_csv(mapping_file, index=False)
                print(f'Custom anomaly added to category: {correct_category}')
            else:
                print('Custom anomaly already exists in the category.')

            # Collect feedback on recommendations
            feedback = input('Were these recommendations helpful? (yes/no): ').strip().lower()
            if feedback == 'yes':
                print('Great! Thanks for your feedback.')
            else:
                print('Thanks for your feedback, we will improve future suggestions.')

    # Check if user wants to continue
    done = input('\nDo you want to enter another anomaly? (yes/no): ').strip().lower()
    if done != 'yes':
        print('Exiting anomaly input.')
        break



Enter your anomaly description: Inconsistent collection specialist assignment
Exact anomaly found. Category: Behavioral & Pattern Deviations

Do you want to enter another anomaly? (yes/no): yes

Enter your anomaly description: Rapid consecutive invoice postings for same customer

Top 5 recommended similar anomalies:
1. "Same reason code used for different reasons" → Category: Reference Codes & Structural Validity
2. "Customer cluster not matching customer segment" → Category: Customer & Master Data Accuracy
3. "Same customer ID but different names/texts" → Category: Customer & Master Data Accuracy
4. "Same customer billed multiple times with duplicate amounts" → Category: Transaction & Value Integrity
5. "Same Terms of Payment Key Text for multiple key codes" → Category: Reference Codes & Structural Validity


KeyboardInterrupt: Interrupted by user